In [ ]:
from ngsolve.meshes import MakeQuadMesh
import math
import ngsolve as ngs
from ngsolve.webgui import Draw

import myassembling
print(myassembling)
print(myassembling.__file__)
print(dir(myassembling))

ngs.SetNumThreads(1)


In [ ]:
Nx, Ny = 4, 4          # number of structured subdomains
nx, ny = 5, 5          # nodes/DoFs per non-overlapping subdomain
ndof_x, ndof_y = Nx * nx, Ny * ny
nelx, nely = ndof_x - 1, ndof_y - 1

mesh = MakeQuadMesh(nelx, nely, mapping=None)
fes = ngs.H1(mesh, order=1, dirichlet="left|right|top|bottom")

assert fes.ndof == ndof_x * ndof_y

In [ ]:
u, v = fes.TnT()
a = ngs.BilinearForm(fes)
grad_u = ngs.grad(u)
grad_v = ngs.grad(v)

# a += (grad_u * grad_u * grad_u * grad_v - f*v) * ngs.dx
a += (grad_u * grad_v + u * u * u * v) * ngs.dx
gfu = ngs.GridFunction(fes)

x, y = ngs.x, ngs.y
gfu.Set((x * (1 - x)) ** 2 + 0.25 * (y * (1 - y)) ** 2)

print("ne =", mesh.ne, "ndof =", fes.ndof)
Draw(gfu, mesh, "u_current")

## Structured DoF Partition And C++ Support Info

`structured_quad_partitions` is a pure row-major DoF partitioner. It does not know about finite elements.

The returned data have the following meaning:

- `nonoverlapping_partition`: disjoint DoF blocks, used for bookkeeping and visualization.
- `overlapping_partition`: expanded DoF blocks, the actual local unknowns passed to C++.

For each overlapping DoF block, C++ builds the support metadata with `BuildLocalSupportInfo(fes, local_dofs)`:

- `info.core_dofs`: local unknowns, equal to the overlapping DoFs.
- `info.support_elements`: integration elements touching the local unknowns.
- `info.support_dofs`: DoFs needed by those support elements.
- `info.core_in_support`: local positions of `core_dofs` inside `support_dofs`.


In [ ]:
from partition import structured_quad_partitions

overlap_size = 1

nonoverlapping_partition, overlapping_partition = structured_quad_partitions(
    Nx,
    Ny,
    nx,
    ny,
    overlap_size=overlap_size,
)

support_infos = [
    myassembling.BuildLocalSupportInfo(fes, local_dofs)
    for local_dofs in overlapping_partition
]

print("overlap_size =", overlap_size)
print("mesh.ne =", mesh.ne)
print("fes.ndof =", fes.ndof)
print("number of subdomains =", len(nonoverlapping_partition))
print("nonoverlapping dof counts =", [len(part) for part in nonoverlapping_partition])
print("overlapping/local dof counts =", [len(part) for part in overlapping_partition])
print("support dof counts =", [len(info.support_dofs) for info in support_infos])
print("support element counts =", [len(info.support_elements) for info in support_infos])

assert len(nonoverlapping_partition) == Nx * Ny
assert len(overlapping_partition) == Nx * Ny
assert len(support_infos) == Nx * Ny

seen = set()
for part in nonoverlapping_partition:
    part_set = set(part)
    assert not (seen & part_set)
    seen.update(part_set)
assert seen == set(range(fes.ndof))

for nonoverlap, overlap, info in zip(nonoverlapping_partition, overlapping_partition, support_infos):
    assert set(nonoverlap).issubset(set(overlap))
    assert list(info.core_dofs) == list(overlap)
    assert set(overlap).issubset(set(info.support_dofs))
    for k, dof in enumerate(info.core_dofs):
        assert info.support_dofs[info.core_in_support[k]] == dof


## Inspect Support Info

The support information below is computed by C++ from the overlapping DoFs. `overlap-extra dofs` are `overlapping_dofs - nonoverlapping_dofs`; `support-extra dofs` are `support_dofs - overlapping_dofs`.


In [ ]:
for i, (nonoverlap, overlap, info) in enumerate(zip(nonoverlapping_partition, overlapping_partition, support_infos)):
    nonoverlap_set = set(nonoverlap)
    overlap_set = set(overlap)
    support_set = set(info.support_dofs)

    print(f"Omega_{i}:")
    print("  nonoverlapping dofs:", len(nonoverlap_set))
    print("  overlapping/local dofs:", len(overlap_set))
    print("  overlap extra dofs:", len(overlap_set - nonoverlap_set))
    print("  support dofs:", len(support_set))
    print("  support extra dofs:", len(support_set - overlap_set))
    print("  support elements:", len(info.support_elements))


## Visualize DoF Layers And Support Elements

Each subplot shows one local problem. The data all come from the same C++ support construction used by the nonlinear operator:

- filled cells: `info.support_elements`, the integration domain
- blue points: non-overlapping DoFs
- red points: overlap-extra DoFs, `overlapping_dofs - nonoverlapping_dofs`
- purple points: support-extra DoFs, `support_dofs - overlapping_dofs`


In [ ]:
import math
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon

volume_elements = list(mesh.Elements(ngs.VOL))

def vertex_xy(vnr):
    p = mesh.vertices[int(vnr)].point
    return float(p[0]), float(p[1])

def element_vertices(elnr):
    return [int(v.nr) for v in volume_elements[int(elnr)].vertices]

def element_polygon(elnr):
    return [vertex_xy(vnr) for vnr in element_vertices(elnr)]

def draw_mesh_edges(ax):
    for el in volume_elements:
        pts = [vertex_xy(v.nr) for v in el.vertices]
        pts.append(pts[0])
        xs, ys = zip(*pts)
        ax.plot(xs, ys, color="0.82", linewidth=0.7, zorder=1)

def add_elements(ax, elements, color="tab:green", alpha=0.22):
    for elnr in elements:
        ax.add_patch(
            Polygon(
                element_polygon(elnr),
                closed=True,
                facecolor=color,
                edgecolor="0.45",
                linewidth=0.45,
                alpha=alpha,
                zorder=2,
            )
        )

def scatter_dofs(ax, dofs, color, label, size=42, zorder=4):
    pts = [vertex_xy(d) for d in sorted(dofs) if int(d) < mesh.nv]
    if pts:
        xs, ys = zip(*pts)
        ax.scatter(xs, ys, s=size, color=color, edgecolor="black", linewidth=0.45, label=label, zorder=zorder)

all_vertex_pts = [vertex_xy(i) for i in range(mesh.nv)]
all_x, all_y = zip(*all_vertex_pts)
pad = 0.04
xmin, xmax = min(all_x) - pad, max(all_x) + pad
ymin, ymax = min(all_y) - pad, max(all_y) + pad

npatches = len(overlapping_partition)
cols = min(4, npatches)
rows = math.ceil(npatches / cols)
fig, axes = plt.subplots(rows, cols, figsize=(3.6 * cols, 3.6 * rows), squeeze=False)

for ax in axes.flat[npatches:]:
    ax.axis("off")

for patch_id, (nonoverlap_part, local_dofs, info) in enumerate(zip(nonoverlapping_partition, overlapping_partition, support_infos)):
    ax = axes.flat[patch_id]
    nonoverlap = set(nonoverlap_part)
    overlap = set(local_dofs)
    support_dofs = set(info.support_dofs)

    assert nonoverlap <= overlap
    assert overlap <= support_dofs
    assert list(info.core_dofs) == list(local_dofs)

    draw_mesh_edges(ax)
    add_elements(ax, info.support_elements)
    ax.scatter(all_x, all_y, s=10, color="0.72", zorder=3)
    scatter_dofs(ax, nonoverlap, "tab:blue", "non-overlap dofs", size=42, zorder=5)
    scatter_dofs(ax, overlap - nonoverlap, "tab:red", "overlap-extra dofs", size=46, zorder=6)
    scatter_dofs(ax, support_dofs - overlap, "tab:purple", "support-extra dofs", size=38, zorder=4)

    ax.set_title(f"Omega_{patch_id}")
    ax.set_aspect("equal")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xticks([])
    ax.set_yticks([])

legend_handles = [
    Patch(facecolor="tab:green", edgecolor="0.45", alpha=0.22, label="support elements"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor="tab:blue", markeredgecolor="black", markersize=7, label="non-overlap dofs"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor="tab:red", markeredgecolor="black", markersize=7, label="overlap-extra dofs"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor="tab:purple", markeredgecolor="black", markersize=7, label="support-extra dofs"),
]
fig.legend(handles=legend_handles, loc="upper center", ncol=4, frameon=True)
plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()


## Global Nonlinear Residual And Jacobian

For verification, assemble the global nonlinear residual and Jacobian using NGSolve's nonlinear form machinery:

- `a.Apply(gfu.vec, res_global)` evaluates the nonlinear residual at the current state.
- `a.AssembleLinearization(gfu.vec)` assembles the Jacobian at the same state, stored as `a.mat`.

No boundary-condition replacement is applied in this demo; the local objects are compared against the raw global nonlinear operator restricted to the overlapping local DoFs.


In [ ]:
# global residual
res_global = gfu.vec.CreateVector()
a.Apply(gfu.vec, res_global)

# global jacobian matrix
a.AssembleLinearization(gfu.vec)
jac_global = a.mat


## Build Local Nonlinear Operators And Evaluate Them

For each subdomain, pass only the overlapping DoFs to C++:

```python
local_op = myassembling.LocalNonlinearOperator(fes, a, local_dofs)
```

The operator uses the same support construction as `BuildLocalSupportInfo`.


In [ ]:
local_operators = []
local_jacobians = []
local_residuals = []

for local_dofs in overlapping_partition:
    local_op = myassembling.LocalNonlinearOperator(fes, a, local_dofs)
    local_operators.append(local_op)
    local_jacobians.append(local_op.Jacobian(gfu.vec))
    local_residuals.append(local_op.Residual(gfu.vec))

for i, (local_dofs, info, jac_local, res_local) in enumerate(zip(overlapping_partition, support_infos, local_jacobians, local_residuals)):
    assert list(res_local.core_dofs) == list(local_dofs)
    assert list(jac_local.core_dofs) == list(local_dofs)
    assert list(res_local.support_dofs) == list(info.support_dofs)
    assert list(res_local.support_elements) == list(info.support_elements)
    print(f"Omega_{i}:")
    print("  nonoverlapping dofs:", len(nonoverlapping_partition[i]))
    print("  overlapping/local dofs:", len(local_dofs))
    print("  support dofs:", len(res_local.support_dofs))
    print("  support elements:", len(res_local.support_elements))
    print("  Jacobian shape:", (jac_local.mat.height, jac_local.mat.width))
    print("  Residual shape:", res_local.vec.size)


## Verify Against Global Nonlinear Assembly

The local nonlinear residual and Jacobian are assembled on the overlapping DoFs. They are compared against the corresponding global restriction:

`F_local == F_global[local_dofs]`

`J_local == J_global[local_dofs, local_dofs]`


In [ ]:
def matrix_entry(mat, i, j):
    return float(mat[int(i), int(j)])

for patch_id, (local_dofs, res_local, jac_local) in enumerate(
    zip(overlapping_partition, local_residuals, local_jacobians)
):
    jac_err = 0.0
    for i, gi in enumerate(local_dofs):
        for j, gj in enumerate(local_dofs):
            jac_err = max(
                jac_err,
                abs(matrix_entry(jac_local.mat, i, j) - matrix_entry(jac_global, gi, gj)),
            )

    res_err = max(
        abs(float(res_local.vec[i]) - float(res_global[d]))
        for i, d in enumerate(local_dofs)
    )

    print(f"Omega_{patch_id}:")
    print("  nonoverlapping dofs:", len(nonoverlapping_partition[patch_id]))
    print("  overlapping/local dofs:", len(local_dofs))
    print("  support dofs:", len(res_local.support_dofs))
    print("  support elements:", len(res_local.support_elements))
    print("  Jacobian inf error:", f"{jac_err:.3e}")
    print("  residual inf error:", f"{res_err:.3e}")

    if not math.isclose(jac_err, 0.0, abs_tol=1e-10):
        raise RuntimeError("local nonlinear Jacobian does not match global local-dof restriction")
    if not math.isclose(res_err, 0.0, abs_tol=1e-10):
        raise RuntimeError("local nonlinear residual does not match global local-dof restriction")
